# Pipeline Analytics

**Kernel:** use `video-benchmark` (launch via `./scripts/notebook.sh`). Do not use the conda `pxt` env.

After code changes, run `run-benchmark --reset` once so catalog columns match the installed package.


In [ ]:
import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pixeltable as pxt

# --- KERNEL CHECK ---
print('python:', sys.executable)
print('pixeltable:', pxt.__version__)
if '.venv' not in sys.executable:
    print('WARNING: select the video-benchmark kernel (.venv), not conda pxt')
if not str(pxt.__version__).startswith('0.7.'):
    print('WARNING: expected pixeltable 0.7.x; run ./scripts/notebook.sh')

pxt.ls('video_benchmarking')


## Pipeline map

Catalog DAG (see [docs/WORKFLOW.md](../docs/WORKFLOW.md)):

`video_sources` → `keyframes` (fps sample when scene-aware) + `audio_chunks` (ASR) → dedupe → select → summarize → (Path 3 compact) → assembled context → synthesis.


In [ ]:
vs = pxt.get_table('video_benchmarking.video_sources')
keyframes = pxt.get_table('video_benchmarking.keyframes')
audio = pxt.get_table('video_benchmarking.audio_chunks')


In [ ]:
vs.describe()


## 1 — Segmentation


In [ ]:
vs.select(vs.scene_cuts, vs.segment_times).tail(1)


In [ ]:
vs.select(vs.segment_times).tail(1)


## 2 — Vision (keyframes)


In [ ]:
keyframes.select(
    keyframes.global_position_ms,
    keyframes.gemini_frame_insight,
    keyframes.oss_frame_insight,
).order_by(keyframes.global_position_ms).limit(8).collect()


In [ ]:
keyframes.select(
    keyframes.frame,
    keyframes.gemini_frame_insight,
    keyframes.oss_frame_insight,
).order_by(keyframes.global_position_ms).limit(1).collect()


## 3 — ASR (audio chunks)


In [ ]:
cols = list(audio.columns)
asr_cols = [c for c in cols if any(k in c for k in (
    'gemini_transcript', 'whisperx_segment', 'whisper_segment', 'gemini_chunk'
))]
print("ASR-related columns:", asr_cols)
sel = [getattr(audio, c) for c in asr_cols[:4]]
asr = audio.select(
    audio.segment_start, *sel
).order_by(audio.segment_start).collect()
print(f"chunks: {len(asr)}")
row0 = asr[0] if asr else {}
for c in asr_cols:
    if c not in row0:
        continue
    v = row0[c]
    if isinstance(v, list):
        print(f"{c}: {len(v)} lines; sample={v[:2]}")
    else:
        preview = str(v)[:200] if v is not None else None
        print(f"{c}: {preview}")


## 4 — Intermediate context (parent table)


In [ ]:
labels = [
    "gemini_frame_context",
    "gemini_frame_context_deduped",
    "gemini_frame_context_selected",
    "gemini_frame_context_summarized",
    "oss_frame_context",
    "oss_frame_context_deduped",
    "oss_frame_context_selected",
    "oss_frame_context_summarized",
    "oss_frame_context_compact",
    "gemini_transcript_context",
    "oss_transcript_context",
]
available = [n for n in labels if hasattr(vs, n)]
ctx_counts = vs.select(
    *[getattr(vs, n) for n in available]
).tail(1).to_pandas().iloc[0]
for name in available:
    val = ctx_counts[name]
    n = len(val) if isinstance(val, list) else 0
    print(f"{name}: {n}")
    if name == "oss_frame_context_compact" and isinstance(val, list) and val:
        chars = [
            len(str(item.get("frame_insight", "")))
            for item in val if isinstance(item, dict)
        ]
        mean_c = sum(chars) / len(chars)
        print(
            f"  compact insight chars: "
            f"min={min(chars)} max={max(chars)} mean={mean_c:.0f}"
        )


## 5 — Assembly (synthesis inputs)


In [ ]:
def _block(text, tag):
    if not text:
        return ""
    m = re.search(rf"<{tag}>(.*?)</{tag}>", text, flags=re.S)
    return (m.group(1).strip() if m else "")

asm = vs.select(
    vs.gemini_orchestrated_context,
    vs.oss_context,
    vs.oss_synthesis_prompt_text,
).tail(1).to_pandas().iloc[0]

for label, col in [
    ("Gemini orchestrated", "gemini_orchestrated_context"),
    ("OSS", "oss_context"),
]:
  text = asm[col] or ""
  audio = _block(text, "audio_transcript")
  visual = _block(text, "visual_keyframe_timeline")
  print(f"=== {label} ===")
  print("audio lines:", len([ln for ln in audio.splitlines() if ln.strip()]))
  print("visual preview:", (visual[:240] + "...") if visual else "N/A")
  print()

prompt = asm["oss_synthesis_prompt_text"] or ""
print("oss_synthesis_prompt_text chars:", len(prompt))
print(prompt[:500], "...")


## 6 — Rollups and cost breakdown


In [ ]:
rollup = vs.select(
    vs.gemini_vision_rollup,
    vs.native_cost,
    vs.gemini_vision_track_cost,
    vs.gemini_asr_cost,
    vs.gemini_synthesis_cost,
    vs.gemini_orchestrated_total,
    vs.oss_cost,
).tail(1).to_pandas().iloc[0]

print("gemini_vision_rollup:", rollup["gemini_vision_rollup"])
print(
    "costs | native:", rollup["native_cost"],
    "| gemini total:", rollup["gemini_orchestrated_total"],
    "| vision:", rollup["gemini_vision_track_cost"],
    "| asr:", rollup["gemini_asr_cost"],
    "| synth:", rollup["gemini_synthesis_cost"],
    "| oss:", rollup["oss_cost"],
)


## 7 — Final insights


In [ ]:
vs.select(
    vs.query,
    vs.video_duration_sec,
    vs.native_insight,
    vs.gemini_orchestrated_insight,
    vs.oss_insight,
    vs.native_cost,
    vs.gemini_orchestrated_total,
    vs.oss_cost,
).tail(1)


In [ ]:
insights = vs.select(
    vs.native_insight,
    vs.gemini_orchestrated_insight,
    vs.oss_insight,
).tail(1)

for col in ['native_insight', 'gemini_orchestrated_insight', 'oss_insight']:
    print('=' * 64)
    print(col)
    print(insights[col][0])
    print()


In [ ]:
# Narrative quality comparison
row = vs.select(
    vs.native_insight,
    vs.gemini_orchestrated_insight,
    vs.oss_insight,
    vs.gemini_transcript_context,
    vs.oss_transcript_context,
    vs.video_duration_sec,
).tail(1).to_pandas().iloc[0]

paths = ["native_insight", "gemini_orchestrated_insight", "oss_insight"]
for col in paths:
    text = str(row[col] or "")
    bullets = text.count("**") // 2
    numbered = sum(
        1 for line in text.splitlines() if line.strip()[:2].rstrip(".").isdigit()
    )
    print(
        f"{col}: {len(text.split())} words, "
        f"{bullets} bold sections, {numbered} numbered lines"
    )
    preview = text[:200].replace(chr(10), " ")
    print(f"  preview: {preview}...")
    print()

print("gemini transcript chunks:", len(row["gemini_transcript_context"] or []))
print("oss transcript chunks:", len(row["oss_transcript_context"] or []))
print("video duration (s):", row["video_duration_sec"])

kf = keyframes.select(keyframes.global_position_ms).order_by(
    keyframes.global_position_ms
).collect()
positions = [r["global_position_ms"] / 1000.0 for r in kf]
print(f"keyframes sampled: {len(positions)}")
if positions:
    fig, ax = plt.subplots(figsize=(8, 2))
    ax.scatter(positions, [1] * len(positions), alpha=0.6)
    ax.set_xlim(0, float(row["video_duration_sec"] or max(positions)))
    ax.set_xlabel("seconds")
    ax.set_title("Keyframe sampling across video timeline")
    ax.set_yticks([])
    plt.tight_layout()
    plt.show()


In [ ]:
costs = vs.select(
    vs.native_cost,
    vs.gemini_orchestrated_total,
    vs.oss_cost,
).tail(1).to_pandas().iloc[0]

fig, ax = plt.subplots(figsize=(7, 4))
labels = ['Native', 'Gemini', 'OSS']
values = [
    costs['native_cost'],
    costs['gemini_orchestrated_total'],
    costs['oss_cost'],
]
ax.bar(labels, values)
ax.set_ylabel('USD')
ax.set_title('Three-path costs')
plt.show()


In [ ]:
export_dirs = sorted(Path('results').glob('*/summary.json'))
if export_dirs:
    latest = export_dirs[-1]
    print("latest export:", latest.parent)
    print(json.loads(latest.read_text()))
else:
    print('No results/ exports yet — run run-benchmark')


In [ ]:
vs.where(vs.native_insight.errortype != None).select(
    vs.query, vs.native_insight.errortype, vs.native_insight.error_msg,
).collect()


In [ ]:
!pxt dashboard
